In [1]:
!pip install pyspark
!pip install findspark

In [6]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
!pip install pyspark
# FindSpark simplifies the process of using Apache Spark with Python
import findspark
findspark.init()

In [7]:

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Simple PySpark App").getOrCreate()



In [8]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv

--2026-02-23 23:18:16--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60682 (59K) [text/csv]
Saving to: ‘NASA_airfoil_noise_raw.csv’

NASA_airfoil_noise_ 100%[===================>]  59.26K   147KB/s    in 0.4s    

2026-02-23 23:18:17 (147 KB/s) - ‘NASA_airfoil_noise_raw.csv’ saved [60682/60682]



In [35]:
df = spark.read.csv("NASA_airfoil_noise_raw.csv", header=True, inferSchema=True)

In [36]:
#task 5
df.show(5)

+---------+-------------+-----------+------------------+-----------------------+----------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevel|
+---------+-------------+-----------+------------------+-----------------------+----------+
|      800|          0.0|     0.3048|              71.3|             0.00266337|   126.201|
|     1000|          0.0|     0.3048|              71.3|             0.00266337|   125.201|
|     1250|          0.0|     0.3048|              71.3|             0.00266337|   125.951|
|     1600|          0.0|     0.3048|              71.3|             0.00266337|   127.591|
|     2000|          0.0|     0.3048|              71.3|             0.00266337|   127.461|
+---------+-------------+-----------+------------------+-----------------------+----------+
only showing top 5 rows


In [37]:
#task 6
rowcount1 = df.count()
print(rowcount1)

1522


In [38]:
#task 7
df = df.dropDuplicates()

In [39]:
#task 8
rowcount2 = df.count()
print(rowcount2)

1503


In [40]:
# task 9
df = df.dropna()


In [42]:
#10
rowcount3 = df.count()
print(rowcount3)

1499


In [43]:
#11
df = df.withColumnRenamed("SoundLevel", "SoundLevelDecibels")

In [44]:
#task 12
df.write.mode("overwrite").parquet("NASA_airfoil_noise_cleaned.parquet")

In [45]:
print("Part 1 - Evaluation")

print("Total rows = ", rowcount1)
print("Total rows after dropping duplicate rows = ", rowcount2)
print("Total rows after dropping duplicate rows and rows with null values = ", rowcount3)
print("New column name = ", df.columns[-1])

import os

print("NASA_airfoil_noise_cleaned.parquet exists :", os.path.isdir("NASA_airfoil_noise_cleaned.parquet"))

Part 1 - Evaluation
Total rows =  1522
Total rows after dropping duplicate rows =  1503
Total rows after dropping duplicate rows and rows with null values =  1499
New column name =  SoundLevelDecibels
NASA_airfoil_noise_cleaned.parquet exists : True


In [68]:

df = spark.read.parquet("NASA_airfoil_noise_cleaned.parquet")


In [69]:
rowcount4 = df.count()
print(rowcount4)

1499


In [70]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.pipeline import PipelineModel

In [71]:
assembler =  VectorAssembler(inputCols=["Frequency", "AngleOfAttack", "ChordLength","FreeStreamVelocity","SuctionSideDisplacement"], outputCol="features")

In [72]:
#stag2
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

In [73]:
#task 5
lr = LinearRegression(featuresCol="scaledFeatures", labelCol="SoundLevelDecibels")

In [74]:
#task 6
pipeline = Pipeline(stages=[ assembler, scaler, lr])


In [75]:
# 7
(trainingData, testingData) = df.randomSplit([0.7, 0.3], seed=42)

In [76]:
#8
pipelineModel = pipeline.fit(trainingData)


In [77]:
print("Part 2 - Evaluation")
print("Total rows = ", rowcount4)
ps = [str(x).split("_")[0] for x in pipeline.getStages()]

print("Pipeline Stage 1 = ", ps[0])
print("Pipeline Stage 2 = ", ps[1])
print("Pipeline Stage 3 = ", ps[2])

print("Label column = ", lr.getLabelCol())

Part 2 - Evaluation
Total rows =  1499
Pipeline Stage 1 =  VectorAssembler
Pipeline Stage 2 =  StandardScaler
Pipeline Stage 3 =  LinearRegression
Label column =  SoundLevelDecibels


In [78]:
#1
predictions = pipelineModel.transform(testingData)

In [79]:
#2
evaluator = RegressionEvaluator(labelCol="SoundLevelDecibels", predictionCol="prediction", metricName="mse")
mse = evaluator.evaluate(predictions)
print(mse)

24.997666255024154


In [80]:
#3
evaluator = RegressionEvaluator(labelCol="SoundLevelDecibels", predictionCol="prediction", metricName="mae")
mae = evaluator.evaluate(predictions)
print(mae)

3.9136790958811947


In [81]:
evaluator = RegressionEvaluator(labelCol="SoundLevelDecibels", predictionCol="prediction", metricName="r2")
r2 = evaluator.evaluate(predictions)
print(r2)

0.49596884089746285


In [82]:
print("Part 3 - Evaluation")

print("Mean Squared Error = ", round(mse,2))
print("Mean Absolute Error = ", round(mae,2))
print("R Squared = ", round(r2,2))

lrModel = pipelineModel.stages[-1]

print("Intercept = ", round(lrModel.intercept,2))


Part 3 - Evaluation
Mean Squared Error =  25.0
Mean Absolute Error =  3.91
R Squared =  0.5
Intercept =  132.88


In [83]:
pipelineModel.write().overwrite().save("Final_Project")

In [84]:
loadedPipelineModel = PipelineModel.load("Final_Project")

In [85]:
#3
predictions =  loadedPipelineModel.transform(testingData)

In [86]:
predictions.select("SoundLevelDecibels", "prediction").show(5)

+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           128.679|122.59722914376775|
|            133.42|127.37968204568844|
|           119.146| 130.3407742507451|
|           116.074|131.11016975113546|
|           134.319|127.12627360125104|
+------------------+------------------+
only showing top 5 rows


In [87]:
print("Part 4 - Evaluation")

loadedmodel = loadedPipelineModel.stages[-1]
totalstages = len(loadedPipelineModel.stages)
inputcolumns = loadedPipelineModel.stages[0].getInputCols()

print("Number of stages in the pipeline = ", totalstages)
for i,j in zip(inputcolumns, loadedmodel.coefficients):
    print(f"Coefficient for {i} is {round(j,4)}")

Part 4 - Evaluation
Number of stages in the pipeline =  3
Coefficient for Frequency is -3.9906
Coefficient for AngleOfAttack is -2.2881
Coefficient for ChordLength is -3.3269
Coefficient for FreeStreamVelocity is 1.4832
Coefficient for SuctionSideDisplacement is -2.0551
